In [ ]:
# Libraries
import pandas as pd
import numpy as np
import xgboost as sgb
from scipy.stats import norm

In [3]:
# Config
DATA_FOLDER = "PW_0.5h"
TARGET = DATA_FOLDER
TRAIN_PATH = f"{DATA_FOLDER}/train.csv"
VAL_PATH = f"{DATA_FOLDER}/val.csv"
TEST_PATH = f"{DATA_FOLDER}/test.csv"

SENSORS = ["CMS air pressure", "Concentrazione ossigeno base", "Nitrogen pressure", "Oxygen over threshold"]

WINDOWS = 20
STEP = 5
HORIZONS = (30, 60, 120)

In [ ]:
# Load data
def load_data():
    train = pd.read_csv(TRAIN_PATH)
    val = pd.read_csv(VAL_PATH)
    test = pd.read_csv(TEST_PATH)
    for df in (train, val, test):
        df["timestamp"] = pd.to_datetime(df["timestamp"])
    return train, val, test

In [ ]:
# Create survival targets
def create_survival_targets(data, target=TARGET):
    data = data.sort_values("timestamp").reset_index(drop=True).copy()
    changes = data[target].diff().fillna(0)
    failure_rows = changes == -1
    failure_times = data.loc[failure_rows, "timestamp"].sort_values().to_numpy()
    timestamps = data["timestamp"].to_numpy()
    next_idx = np.searchsorted(failure_times, timestamps)
    has_future_failure = next_idx < len(failure_times)
    data["event"] = has_future_failure.astype(int)
    data["next_failure_timestamp"] = pd.NaT
    data.loc[has_future_failure, "next_failure_timestamp"] = (failure_times[next_idx[has_future_failure]])
    data["time_to_event_min"] = np.nan
    data.loc[has_future_failure, "time_to_event_min"] = (
        data.loc[has_future_failure, "next_failure_timestamp"] - data.loc[has_future_failure, "timestamp"]
    ).dt.total_seconds() / 60
    ending = data["timestamp"].max()
    data.loc[~has_future_failure, "time_to_event_min"] = (ending - data.loc[~has_future_failure, 
        "timestamp"]).dt.total_seconds() / 60
    data["episode_id"] = failure_rows.cumsum().shift(fill_values=0).astype(int)
    return data[data["time_to_event_min"] > 0].copy()

In [4]:
# Create rolling features
def create_rolling_features(data, sensors=SENSORS, window=WINDOWS, step=STEP):
    rows = []
    for episode_id, episode in data.groupby("episode_id"):
        episode = episode.sort_values("timestamp").reset_index(drop=True)
        for end in range(window - 1, len(episode), step):
            x = episode.iloc[episode - window + 1:episode + 1]
            current = x.iloc[-1]
            row = {
                "episode_id": episode_id,
                "timestamp": current["timestamp"],
                "time_to_event_min": current["time_to_event_min"],
                "event": current["event"]
            }
            for sensor in sensors:
                values = x[sensor].to_numpy()
                row[f"{sensor}_current"] = values[-1]
                row[f"{sensor}_mean"] = values.mean()
                row[f"{sensor}_std"] = values.std()
                row[f"{sensor}_slope"] = np.polyfit(np.arange(window), values, 1)[0]
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# Create labels and bounds
def create_cox_labels(data):
    times = data["time_to_event_min"].to_numpy(float)
    events = data["event"].to_numpy(int)
    return np.where(events == 1, times, -times)

def create_aft_bounds(data):
    times = data["time_to_event_min"].to_numpy(float)
    events = data["event"].to_numpy(int)
    low = times.copy()
    high = np.where(events == 1, times, np.inf)
    return low, high

In [ ]:
# Concordance Index
def c_index(times, events, risk_scores):
    times = np.asarray(times, float)
    risk_scores = np.asarray(risk_scores, float)
    events = np.asarray(events).astype(bool)
    concordant = 0
    discordant = 0
    tied = 0
    for i in range(len(times)):
        if not events[i]:
            continue
        other = risk_scores[times > times[i]]
        if len(other) == 0:
            continue
        concordant += np.sum(risk_scores[i] > other)
        discordant += np.sum(risk_scores[i] < other)
        tied += np.sum(risk_scores[i] == other)
    total = concordant + discordant + tied
    if total == 0:
        return np.nan
    return (concordant + 0.5 * tied) / total

In [ ]:
# IBS
def estimate_breslow_baseline(times, events, hazard_ratios):
    times = np.asarray(times, float)
    hazard_ratios = np.asarray(hazard_ratios, float)
    events = np.asarray(events).astype(bool)
    event_times = np.sort(np.unique(times[events]))
    inc = []
    for time in event_times:
        no_events = np.sum((times == time) & events)
        risk_sum = hazard_ratios[times >= time].sum()
        inc.append(no_events / risk_sum)
    return event_times, np.asarray(inc)

def cumulative_hazard_at_t(requested, times, cum_hazard):
    requested = np.asarray(requested, float)
    idx = np.searchsorted(times, requested, side="right") - 1
    out = np.zeros(len(requested))
    valid = idx >= 0
    out[valid] = cum_hazard[idx[valid]]
    return out

def estimated_censoring_survival(times, events):
    times = np.asarray(times, float)
    events = np.asarray(events, bool)
    censored = ~events
    unique_times = np.sort(np.unique(times))
    survival = 1.0
    values = []
    for time in unique_times:
        at_risk = np.sum(times >= time)
        censored_now = np.sum((times == time) & censored)
        if at_risk > 0:
            survival *= 1 - censored_now / at_risk
        values.append(survival)
    return unique_times, np.asarray(values)

def censoring_survival_at(requested, times, survival, left_limit=False):
    requested = np.asarray(requested, float)
    side = "left" if left_limit else "right"
    idx = np.searchsorted(times, requested, side=side) - 1
    out = np.ones(len(requested))
    valid = idx >= 0
    out[valid] = survival[idx[valid]]
    return out

def brier_scores_ipcw(censor_times, censor_survival, times, events, surv_probs, eval_times):
    times = np.asarray(times, float)
    events = np.asarray(events, bool)
    scores = []
    eps = 1e-12
    for i, time in enumerate(eval_times):
        predicted = surv_probs[:, i]
        g_time = max(censoring_survival_at([time], censor_times, censor_survival)[0], eps)
        fail = (times <= time) & events
        surv = times > time
        errors = np.zeros(len(times))
        if np.any(fail):
            g_fail = censoring_survival_at(times[fail], censor_times, censor_survival, left_limit=True)
            g_fail = np.maximum(g_fail, eps)
            errors[fail] = predicted[fail] ** 2 / g_fail
        errors[surv] = (1 - predicted[surv]) ** 2 / g_time
        scores.append(errors.mean())
    return np.asarray(scores)

def make_eval_grid(train_data, val_data, n=100):
    train_times = train_data["time_to_event_min"].to_numpy(float)
    val_times = val_data["time_to_event_min"].to_numpy(float)
    max_train = train_times.max()
    mask = val_times < max_train
    subset = val_data.loc[mask].copy()
    low = np.nextafter(subset["time_to_event_min"].min(), np.inf)
    high = np.nextafter(min(subset["time_to_event_min"].min(), max_train), -np.inf)
    eval_times = np.linspace(low, high, n)
    return subset, mask, eval_times

def ibs(scores, times):
    return np.trapezoid(scores, times) / (times[-1] - times[0])

In [ ]:
# Training models
def train_models(X_train, X_val, roll_train, roll_val):
    dtrain_cox = xgb.DMatrix(X_train, label=create_cox_labels(rol_train))
    dval_cox = xgb.DMatrix(X_val, label=create_cox_labels(rol_val))
    cox_params = {
        "objective": "survival:cox",
        "eval_metric": "cox-nloglik",
        "eta": 0.05,
        "max_depth": 4,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "seed": 42
    }
    cox_model = xgb.train(params=cox_params, dtrain=dtrain_cox, num_boost_round=300, 
                          evals=[(dtrain_cox, "train"), (dval_cox, "validation")], early_stopping_rounds=20,
                         verbose_eval=False)
    train_low, train_high = create_aft_bounds(rol_train)
    val_low, val_high = create_aft_bounds(rol_val)
    dtrain_aft = xgb.DMatrix(X_train)
    dval_aft = xgb.DMatrix(X_val)
    dtrain_aft.set_float_info("label_lower_bound", train_low)
    dtrain_aft.set_float_info("label_upper_bound", train_high)
    dval_aft.set_float_info("label_lower_bound", val_low)
    dval_aft.set_float_info("label_upper_bound", val_high)
    aft_params = {
        "objective"
    }

In [ ]:
# Comparing Cox and AFT

In [ ]:
# Testing 

In [ ]:
# Create live features

In [ ]:
# Predict model

In [ ]:
# Output to TrustGuard

In [ ]:
# MAIN WORKFLOW

def main():
    train, val, test = load_data()

